In [0]:
#Leyendo la tabla de caracteristicas generales
ventas = spark.read.format("delta").table("workspace.electiva.caso_practico_ventas")

# Convertir el DataFrame de Spark a Pandas
ventas = ventas.toPandas()

ventas

     

**FASE DE TRABAJO 1 Y 2**

**BLOQUE 1**

**_IDENTIFICACION DE TIPOS_**

In [0]:
ventas.dtypes

In [0]:
# Convertir la columna id_transaccion a tipo texto (string)
ventas['id_transaccion'] = ventas['id_transaccion'].astype('string')

# Convertir la columna id_cliente a tipo texto (string)
ventas['id_cliente'] = ventas['id_cliente'].astype('string')

# Convertir la columna categoria_producto a tipo texto (categorico)
ventas['categoria_producto'] = ventas['categoria_producto'].astype('category')

# Convertir la columna metodo_pago a tipo texto (categorico)
ventas['metodo_pago'] = ventas['metodo_pago'].astype('category')

# Convertir la columna region a tipo texto (categorico)
ventas['region'] = ventas['region'].astype('category')
ventas.dtypes

In [0]:
# Convertir la columna cantidad a tipo entero (int)
ventas['cantidad'] = ventas['cantidad'].fillna(0).astype('int')
ventas.dtypes

In [0]:
# Convertir la columna satisfaccion_cliente a tipo entero (int) y limitar valores del 1 al 5
ventas['satisfaccion_cliente'] = ventas['satisfaccion_cliente'].fillna(0).astype('int')

# Filtrar solo los valores que estan entre 1 y 5
ventas = ventas[(ventas['satisfaccion_cliente'] >= 1) & (ventas['satisfaccion_cliente'] <= 5)]

ventas.dtypes

In [0]:
import pandas as pd
(ventas['fecha'])=pd.to_datetime(ventas['fecha'])

In [0]:
ventas.dtypes

**_DETECCION DE NULOS_**

In [0]:
ventas.info() 

**_ESTADISTICA DESCRIPTIVA_**

In [0]:
ventas[['cantidad', 'precio_unitario']].describe()

**BLOQUE 2**

**_TRATAMIENTO DE DUPLICADOS_**

In [0]:
ventas = ventas.drop_duplicates(subset=['id_transaccion'], keep='first')

**NORMALIZACION DE TEXTO**

In [0]:
# Normalizar texto en la columna metodo_pago: unificar variantes de 'tarjeta credito'
ventas['metodo_pago'].astype('string')
ventas['metodo_pago'].str.strip().str.lower()
# Quitar tildes para unificar variantes acentuadas
ventas['metodo_pago'].str.normalize('NFKD').str.encode('ascii', errors='ignore').str.decode('utf-8')
ventas['metodo_pago'].replace({'tarjeta Credito	': 'Tarjeta de Crédito'})
# Restaurar a tipo categorico
ventas['metodo_pago'].astype('category')
# Mostrar los valores unicos despues de la normalizacion
ventas['metodo_pago'].unique()

**_GESTION DE VALORES NULOS_**

In [0]:
# Reemplazar los valores nulos en la columna cantidad por la mediana de la columna
ventas['cantidad'].fillna(ventas['cantidad'].median())



In [0]:
# Imputar los faltantes en satisfaccion_cliente usando la moda (valor mas frecuente)
ventas['satisfaccion_cliente'].fillna(ventas['satisfaccion_cliente'].mean())


**_TRATAMIENTO DE ATIPICOS (Outliers):_**

In [0]:
ventas['precio_unitario'].fillna(9999.00)

**FASE DE TRABAJO 3, 4 Y 5**

**BLOQUE 3**

**_COLUMNA DERIVADA_**

In [0]:
ventas['monto_total'] = ventas['cantidad'] * ventas['precio_unitario']
ventas


**_CONVERSION TEMPORAL_**

In [0]:
pd.to_datetime(ventas['fecha'])
ventas['fecha'].dt.day_name()

**_DISCRETIZACION (Binning):_**

In [0]:
ventas['rango_monto'] = pd.cut(
    ventas['monto_total'],
    bins=[-float('inf'), 100, 500, float('inf')],
    labels=['Bajo', 'Medio', 'Alto']
)

In [0]:
ventas['rango_monto']

**BLOQUE 4**

**_AGRUPAMIENTO SIMPLE GROUP BY_**

In [0]:
ventas.groupby('categoria_producto')['monto_total'].agg('sum')

_**TABLA DINAMICA pivot table**_

In [0]:
pd.pivot_table(ventas, values='monto_total', index='region', columns='categoria_producto')

**_ANALISIS DE SATISFACCION_**

In [0]:
ventas.groupby('metodo_pago')['satisfaccion_cliente'].median()

**BLOQUE 5**

**_FILTRADO MULTICRITERIO_**

In [0]:
ventas[(ventas['region']== 'Norte') & (ventas['satisfaccion_cliente']  >=4)]

**_MUESTREO ESTRATIFICADO_**

In [0]:
ventas.groupby('region').sample(frac=0.10)